In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score
)

In [2]:
df = pd.read_csv("Well production analysis data.csv")

In [3]:
#preprocessing

df["DATEPRD"] = pd.to_datetime(df["DATEPRD"], format="%d-%b-%y")

numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

df["FLOW_KIND"] = df["FLOW_KIND"].fillna("Unknown")

In [4]:
#feature engineering

df["Pressure_Diff"] = df["AVG_DOWNHOLE_PRESSURE"] - df["AVG_WHP_P"]

df["Temp_Diff"] = df["AVG_DOWNHOLE_TEMPERATURE"] - df["AVG_WHT_P"]

In [5]:
features = [
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_DP_TUBING",
    "AVG_ANNULUS_PRESS",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "Pressure_Diff",
    "Temp_Diff"
]

X = df[features]

y = df["BORE_OIL_VOL"]

In [6]:
# ---- Dataset Overview: Understanding X and y ----

print("Dataset shape:", df.shape)
print("\nSample rows:\n")
display(df[features + ["BORE_OIL_VOL"]].sample(5, random_state=42))

print("\nSummary statistics for selected features and target:\n")
display(df[features + ["BORE_OIL_VOL"]].describe())

print("""
Why these X and y values?

Target (y): BORE_OIL_VOL
  This is the volume of oil produced by the well — the quantity we ultimately
  want to predict/forecast. It's the business outcome of interest, so it's
  the natural regression target.

Features (X): sensor/operational readings that physically drive oil flow
  - ON_STREAM_HRS            : hours the well was actively producing —
                                 more active time generally means more oil.
  - AVG_DOWNHOLE_PRESSURE/TEMP: conditions at the reservoir depth, which
                                 govern how easily oil can flow into the well.
  - AVG_DP_TUBING             : pressure drop across the tubing, reflecting
                                 flow resistance on the way to the surface.
  - AVG_ANNULUS_PRESS         : annulus pressure, another indicator of
                                 downhole flow dynamics.
  - AVG_CHOKE_SIZE_P          : the choke setting, which directly controls
                                 how much flow is allowed to surface.
  - AVG_WHP_P / AVG_WHT_P     : wellhead pressure/temperature, surface-level
                                 readings that respond to reservoir output.
  - Pressure_Diff / Temp_Diff : engineered features capturing the pressure
                                 and temperature drop between downhole and
                                 wellhead — a proxy for flow resistance and
                                 energy loss along the wellbore.

In short: X is every measurable operating condition that physically
influences how much oil reaches the surface, and y is the oil volume
outcome we're trying to explain/predict from those conditions.
""")

Dataset shape: (15634, 18)

Sample rows:



,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,Pressure_Diff,Temp_Diff,BORE_OIL_VOL
15243,24.0,232.897,103.1865,175.589,16.3085,52.09688,37.934,80.0715,194.963,23.115,558.0
9097,24.0,232.897,103.1865,175.589,16.3085,52.09688,37.934,80.0715,194.963,23.115,558.0
5729,24.0,235.974,106.3080,183.635,16.3085,48.26827,52.338,89.1420,183.636,17.166,2764.0
8957,0.0,345.907,90.0340,345.907,0.0000,1.25701,0.000,0.0000,345.907,90.034,0.0
10345,24.0,232.897,103.1865,175.589,16.3085,52.09688,37.934,80.0715,194.963,23.115,558.0



Summary statistics for selected features and target:



,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,Pressure_Diff,Temp_Diff,BORE_OIL_VOL
count,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000,15634.000000
mean,20.067196,203.549660,88.238867,163.205050,15.575518,53.849220,42.292969,72.850723,161.256691,15.388144,873.037866
std,8.310563,86.900537,36.917633,59.136878,6.016015,27.755406,19.292842,22.055744,85.125601,42.665536,1047.167379
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-137.311000,-92.459000,0.000000
25%,24.000000,222.590250,101.109000,169.643500,16.223500,47.593885,33.894000,77.511250,169.546750,15.332750,399.000000
50%,24.000000,232.897000,103.186500,175.589000,16.308500,52.096880,37.934000,80.071500,194.963000,23.115000,558.000000
75%,24.000000,241.305750,105.031000,181.407500,16.466000,60.656205,41.293750,82.439250,194.963000,25.317750,747.000000
max,25.000000,397.589000,108.502000,345.907000,30.020000,100.000000,137.311000,93.510000,345.907000,103.186500,5902.000000



Why these X and y values?

Target (y): BORE_OIL_VOL
  This is the volume of oil produced by the well — the quantity we ultimately
  want to predict/forecast. It's the business outcome of interest, so it's
  the natural regression target.

Features (X): sensor/operational readings that physically drive oil flow
  - ON_STREAM_HRS            : hours the well was actively producing —
                                 more active time generally means more oil.
  - AVG_DOWNHOLE_PRESSURE/TEMP: conditions at the reservoir depth, which
                                 govern how easily oil can flow into the well.
  - AVG_DP_TUBING             : pressure drop across the tubing, reflecting
                                 flow resistance on the way to the surface.
  - AVG_ANNULUS_PRESS         : annulus pressure, another indicator of
                                 downhole flow dynamics.
  - AVG_CHOKE_SIZE_P          : the choke setting, which directly controls
                                 

In [ ]:
#supervised
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Features
features = [
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_DP_TUBING",
    "AVG_ANNULUS_PRESS",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "Pressure_Diff",
    "Temp_Diff"
]

X = df[features]
y = df["BORE_OIL_VOL"]

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR()
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 Score": r2_score(y_test, pred)
    })

results = pd.DataFrame(results)

results = results.sort_values("R2 Score", ascending=False)

print(results)

In [ ]:
#Comparison Plot

fig, ax = plt.subplots(figsize=(10,6))

x = np.arange(len(results))

width = 0.25

ax.bar(x-width, results["MAE"], width, label="MAE")
ax.bar(x, results["RMSE"], width, label="RMSE")
ax.bar(x+width, results["R2 Score"], width, label="R2")

ax.set_xticks(x)
ax.set_xticklabels(results["Model"], rotation=30)

plt.legend()

plt.title("Comparison of Regression Models")

plt.show()

In [ ]:
#Unsupervised Comparison
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

cluster_features = [
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "BORE_OIL_VOL"
]

cluster_data = StandardScaler().fit_transform(df[cluster_features])

algorithms = {
    "KMeans": KMeans(n_clusters=3, random_state=42),
    "Agglomerative": AgglomerativeClustering(n_clusters=3),
    "DBSCAN": DBSCAN(eps=0.8, min_samples=10)
}

cluster_results = []

for name, algo in algorithms.items():

    labels = algo.fit_predict(cluster_data)

    if len(set(labels)) > 1 and -1 not in set(labels):

        score = silhouette_score(cluster_data, labels)

    else:

        score = np.nan

    cluster_results.append([name, score])

cluster_results = pd.DataFrame(
    cluster_results,
    columns=["Algorithm","Silhouette Score"]
)

print(cluster_results)

In [ ]:
plt.figure(figsize=(6,5))

plt.bar(
    cluster_results["Algorithm"],
    cluster_results["Silhouette Score"]
)

plt.ylabel("Silhouette Score")

plt.title("Comparison of Clustering Algorithms")

plt.show()